# Let's practice SQL

In [130]:
# Загрузим необходимые библиеки
import duckdb
import pandas as pd
import numpy as np
import re, io
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO

In [131]:
# In-memory connection
conn = duckdb.connect(":memory:")

# Load from CSV directly into an in-memory DuckDB table
conn.execute("""--sql
    CREATE TABLE salaries AS
    SELECT * FROM read_csv_auto('salaries_clean.csv')
""")

# Quick check
print(conn.execute("SELECT COUNT(*) FROM salaries").fetchall())
print(conn.execute("SELECT * FROM salaries LIMIT 5").df())

[(1366,)]
    Salary  Quarter  Year  Age Gender  Started          Profession   Grade  \
0   800000        3  2025   29   Male   2022.0   Backend Developer  Middle   
1   500000        3  2025   19   Male   2024.0   Backend Developer  Junior   
2  1160000        1  2025   23   Male   2022.0  Frontend Developer  Senior   
3   500000        3  2025   20   Male   2023.0   Backend Developer  Junior   
4  1200000        3  2025   33   Male   2017.0   Backend Developer  Middle   

  Localization    City       Skill                                 Industry  \
0        Local  Almaty         PHP                               IT продукт   
1      Foreign  Almaty        Java                               IT продукт   
2        Local  Almaty  Javascript                               E-Commerce   
3        Local  Astana        Java  Государственное учреждение (Government)   
4        Local  Almaty         PHP                               IT продукт   

              Created  
0 2025-09-01 17:08:55 

In [132]:
# top 3 highest payed professions
conn.execute("""--sql--sql
    SELECT *
    FROM salaries
    ORDER BY Salary DESC
    LIMIT 3
""").df()

,Salary,Quarter,Year,Age,Gender,Started,Profession,Grade,Localization,City,Skill,Industry,Created
0,5400000,1,2025,34,Male,2014.0,Системный аналитик | System Analyst,Senior,Local,Almaty,None,Финтех,2025-05-22 04:22:08
1,4000000,2,2025,29,Male,2018.0,Product Manager,Lead,Foreign,Almaty,None,IT продукт,2025-06-04 08:37:21
2,4000000,2,2025,29,Male,2019.0,Chief Technical Officer (CTO),Lead,Local,Almaty,Golang,Стартап,2025-04-09 10:26:07


In [133]:
# top 3 lowest payed professions
conn.execute("""--sql--sql
    SELECT *
    FROM salaries
    ORDER BY Salary ASC
    LIMIT 3
""").df()

,Salary,Quarter,Year,Age,Gender,Started,Profession,Grade,Localization,City,Skill,Industry,Created
0,58000,4,2024,24,Male,NaN,Дата инженер | Data Engineer,Middle,Local,Almaty,None,Банк,2024-10-11 13:05:57
1,75000,1,2025,27,Male,NaN,Android Developer,Trainee,Local,Almaty,None,E-Commerce,2025-02-05 09:20:51
2,75000,1,2025,20,Male,NaN,Copywriter,Trainee,Local,Aktobe,None,Продажи и маркетинг,2025-02-05 08:51:23


In [134]:
# let's count professions by gender
conn.execute("""--sql--sql
    SELECT 
        Gender, 
        COUNT(Profession) AS Count
    FROM salaries
    Group by Gender
    ORDER BY Count DESC
""").df()

,Gender,Count
0,Male,1078
1,Female,259
2,PreferNotToSay,29


In [135]:
# let's count distinct professions by gender
conn.execute("""--sql--sql
    SELECT 
        Gender, 
        COUNT(DISTINCT Profession) AS Count
    FROM salaries
    Group by Gender
    ORDER BY Count DESC
""").df()

,Gender,Count
0,Male,58
1,Female,36
2,PreferNotToSay,8


In [136]:
# Retrieve all professions who joined in 2025
conn.execute("""--sql--sql
    SELECT *
    FROM salaries
    WHERE Year = 2025
    LIMIT 10
""").df()

,Salary,Quarter,Year,Age,Gender,Started,Profession,Grade,Localization,City,Skill,Industry,Created
0,800000,3,2025,29,Male,2022.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,2025-09-01 17:08:55
1,500000,3,2025,19,Male,2024.0,Backend Developer,Junior,Foreign,Almaty,Java,IT продукт,2025-08-29 20:02:53
2,1160000,1,2025,23,Male,2022.0,Frontend Developer,Senior,Local,Almaty,Javascript,E-Commerce,2025-08-28 15:57:36
3,500000,3,2025,20,Male,2023.0,Backend Developer,Junior,Local,Astana,Java,Государственное учреждение (Government),2025-08-28 11:44:12
4,1200000,3,2025,33,Male,2017.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,2025-08-28 06:07:45
5,960000,3,2025,24,Male,2024.0,Quality Assurance (QA),Middle,Local,Almaty,Java,Авиа-компания,2025-08-27 20:19:52
6,1000000,3,2025,22,Male,2022.0,Backend Developer,Middle,Local,Almaty,Golang,Продажи и маркетинг,2025-08-27 14:56:20
7,1200000,3,2025,29,Male,2022.0,Frontend Developer,Middle,Local,Astana,React,Энергетика,2025-08-27 09:29:46
8,500000,3,2025,24,Male,2023.0,Системный аналитик | System Analyst,Junior,Local,Astana,None,Государственное учреждение (Government),2025-08-27 08:39:59
9,1310000,3,2025,21,Male,2023.0,Backend Developer,Middle,Foreign,Almaty,Golang,IT продукт,2025-08-27 07:02:48


In [137]:
# Count the number of entries per city and sort in descending order
conn.execute("""--sql--sql
    SELECT 
        City, 
        COUNT(*) AS Count
    FROM salaries
    GROUP BY City
    ORDER BY Count DESC
""").df()

,City,Count
0,Almaty,901
1,Astana,396
2,Karaganda,20
3,Aktau,8
4,Kostanay,8
5,Shymkent,7
6,Aktobe,5
7,Atyrau,5
8,Oral,3
9,Taraz,3


In [138]:
# Calculate average salary per profession using DuckDB
conn.execute("""--sql--sql
    SELECT 
        Profession, 
        CAST(AVG(Salary) AS INT) AS Avg_Salary
    FROM salaries
    GROUP BY Profession
    ORDER BY Avg_Salary DESC
    LIMIT 10
""").df()

,Profession,Avg_Salary
0,Security Engineer,3290000
1,AI Engineer,2600000
2,Архитектор (Architect),2225000
3,CVM аналитик,2000000
4,Head Of Department,1960159
5,Chief Technical Officer (CTO),1904091
6,Скрам-мастер (Scrum Master),1900000
7,Delivery Manager,1866667
8,Техлид (Techleader),1855667
9,Chief Product Officer (CPO),1810000


In [139]:
# Calculate median salary per profession using DuckDB
conn.execute("""--sql--sql
    SELECT 
        Profession, 
        CAST(median(Salary) as int) AS Median_Salary
    FROM salaries
    GROUP BY Profession
    ORDER BY Median_Salary DESC
    LIMIT 10
""").df()

,Profession,Median_Salary
0,Security Engineer,3290000
1,AI Engineer,2600000
2,Архитектор (Architect),2550000
3,Chief Technical Officer (CTO),2000000
4,CVM аналитик,2000000
5,Computer vision engineer,2000000
6,Delivery Manager,1900000
7,Скрам-мастер (Scrum Master),1900000
8,Head Of Department,1900000
9,Техлид (Techleader),1850000


In [140]:
# Calculate average salary per grade using DuckDB
conn.execute("""--sql--sql
    SELECT 
        Grade, 
        CAST(AVG(Salary) AS INT) AS Avg_Salary
    FROM salaries
    GROUP BY Grade
    ORDER BY Avg_Salary DESC
""").df()

,Grade,Avg_Salary
0,Lead,1648205
1,Senior,1452603
2,Middle,884842
3,Junior,395395
4,Trainee,199417


In [141]:
# Calculate median salary per grade using DuckDB
conn.execute("""--sql
    SELECT 
        Grade, 
        CAST(median(Salary) AS INT) AS Median_Salary
    FROM salaries
    GROUP BY Grade
    ORDER BY Median_Salary DESC
""").df()

,Grade,Median_Salary
0,Lead,1500000
1,Senior,1292500
2,Middle,800000
3,Junior,350000
4,Trainee,175000


In [142]:
# Calculate average salary per city using DuckDB
conn.execute("""--sql
    SELECT 
        City, 
        CAST(AVG(Salary) AS INT) AS Avg_Salary
    FROM salaries
    GROUP BY City
    ORDER BY Avg_Salary DESC
""").df()

,City,Avg_Salary
0,Oral,1100000
1,Saran,1100000
2,Almaty,1069225
3,Kostanay,1000000
4,Astana,879791
5,Petropavl,875000
6,Karaganda,874465
7,Taraz,718333
8,Shymkent,621429
9,Aktau,566736


In [143]:
# Calculate average salary per gender using DuckDB
conn.execute("""--sql
    SELECT 
        Gender, 
        CAST(AVG(Salary) AS INT) AS Avg_Salary
    FROM salaries
    GROUP BY Gender
    ORDER BY Avg_Salary DESC
""").df()

,Gender,Avg_Salary
0,PreferNotToSay,1074069
1,Male,1031645
2,Female,838421


In [144]:
# get latest entry for each profession
conn.execute("""--sql
    SELECT 
        Profession, 
        Salary, 
        Grade, 
        max(Created) AS Latest_Entry
    FROM salaries
    Group By Profession, Salary, Grade
    Order By Latest_Entry DESC
    Limit 5
""").df()

,Profession,Salary,Grade,Latest_Entry
0,Backend Developer,800000,Middle,2025-09-01 17:08:55
1,Backend Developer,500000,Junior,2025-08-29 20:02:53
2,Frontend Developer,1160000,Senior,2025-08-28 15:57:36
3,Backend Developer,1200000,Middle,2025-08-28 06:07:45
4,Quality Assurance (QA),960000,Middle,2025-08-27 20:19:52


In [145]:
# Calculate average salary per Localization using DuckDB
conn.execute("""--sql--sql
    SELECT 
        Localization,
        COUNT(*) AS Count,
        CAST(AVG(Salary) AS INT) AS Avg_Salary
    FROM salaries
    GROUP BY Localization
    ORDER BY Avg_Salary DESC
""").df()

,Localization,Count,Avg_Salary
0,Foreign,205,1546516
1,Local,1161,898687


In [146]:
# Calculate median salary per Localization using DuckDB
conn.execute("""--sql
    SELECT 
        Localization, 
        CAST(median(Salary) AS INT) AS Median_Salary
    FROM salaries
    GROUP BY Localization
    ORDER BY Median_Salary DESC
""").df()

,Localization,Median_Salary
0,Foreign,1400000
1,Local,800000


In [147]:
# get oldest entry for each profession
conn.execute("""--sql--sql
    SELECT
        Profession,
        min(Created) AS Oldest_Entry
    FROM salaries
    GROUP BY Profession
    ORDER BY Oldest_Entry ASC\
    Limit 5
""").df()

,Profession,Oldest_Entry
0,Backend Developer,2024-09-04 10:04:53
1,Project Manager,2024-09-04 17:29:37
2,Chief Technical Officer (CTO),2024-09-04 17:30:40
3,Дата аналитик | Data Analyst,2024-09-04 17:31:43
4,Product Analyst,2024-09-04 17:42:44


In [148]:
# Get the most frequently appeared profession using DuckDB
conn.execute("""--sql--sql
    SELECT 
        Profession, 
        COUNT(*) AS Count
    FROM salaries
    GROUP BY Profession
    ORDER BY Count DESC
    Limit 5
""").df()

,Profession,Count
0,Backend Developer,280
1,Frontend Developer,164
2,Дата аналитик | Data Analyst,103
3,Quality Assurance (QA),64
4,Дата инженер | Data Engineer,63


In [149]:
# Load the salaries table from DuckDB into a pandas DataFrame
df = conn.execute("SELECT * FROM salaries").df()
df.head()

,Salary,Quarter,Year,Age,Gender,Started,Profession,Grade,Localization,City,Skill,Industry,Created
0,800000,3,2025,29,Male,2022.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,2025-09-01 17:08:55
1,500000,3,2025,19,Male,2024.0,Backend Developer,Junior,Foreign,Almaty,Java,IT продукт,2025-08-29 20:02:53
2,1160000,1,2025,23,Male,2022.0,Frontend Developer,Senior,Local,Almaty,Javascript,E-Commerce,2025-08-28 15:57:36
3,500000,3,2025,20,Male,2023.0,Backend Developer,Junior,Local,Astana,Java,Государственное учреждение (Government),2025-08-28 11:44:12
4,1200000,3,2025,33,Male,2017.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,2025-08-28 06:07:45


In [150]:
# Count the number of entries with NULL skills
conn.execute("""--sql--sql
    SELECT count(1)
    FROM salaries
    WHERE Skill IS NULL
""").df()

,count(1)
0,487


In [151]:
# Count distinct skills in every profession using DuckDB
conn.execute("""--sql--sql
    SELECT 
        Profession, 
        COUNT(DISTINCT Skill) AS Distinct_Skills
    FROM salaries
    GROUP BY Profession
    ORDER BY Distinct_Skills DESC
    Limit 5
""").df()

,Profession,Distinct_Skills
0,Backend Developer,19
1,Fullstack Developer,15
2,Quality Assurance (QA),13
3,Тимлид | Teamleader,12
4,Frontend Developer,10


In [152]:
# Show top 10 paid professions using DuckDB
conn.execute("""--sql--sql
    SELECT *
    FROM salaries
    ORDER BY Salary DESC
    LIMIT 10
""").df()

,Salary,Quarter,Year,Age,Gender,Started,Profession,Grade,Localization,City,Skill,Industry,Created
0,5400000,1,2025,34,Male,2014.0,Системный аналитик | System Analyst,Senior,Local,Almaty,None,Финтех,2025-05-22 04:22:08
1,4000000,2,2025,29,Male,2018.0,Product Manager,Lead,Foreign,Almaty,None,IT продукт,2025-06-04 08:37:21
2,4000000,2,2025,29,Male,2019.0,Chief Technical Officer (CTO),Lead,Local,Almaty,Golang,Стартап,2025-04-09 10:26:07
3,4000000,1,2025,25,Male,NaN,Backend Developer,Middle,Foreign,Almaty,Java,IT продукт,2025-02-05 13:30:14
4,3667300,1,2025,31,Male,2015.0,Backend Developer,Senior,Foreign,Almaty,.NET,Продажи и маркетинг,2025-02-03 08:12:55
5,3664238,4,2024,29,Male,NaN,Frontend Developer,Senior,Foreign,Almaty,Typescript,IT продукт,2024-12-28 08:29:06
6,3500000,1,2025,40,Male,2007.0,Backend Developer,Senior,Foreign,Almaty,Golang,E-Commerce,2025-06-18 08:28:54
7,3500000,2,2025,37,Male,2010.0,Backend Developer,Senior,Foreign,Astana,PHP,Финтех,2025-05-16 13:52:11
8,3500000,1,2025,25,Male,NaN,Backend Developer,Middle,Foreign,Almaty,Java,IT продукт,2025-01-24 17:25:00
9,3500000,4,2024,36,Male,NaN,Head Of Department,Lead,Local,Almaty,Дата аналитика | Data Analysis,IT продукт,2024-10-11 13:05:57


In [153]:
# Show least 10 salaries using DuckDB
conn.execute("""--sql
    SELECT *
    FROM salaries
    ORDER BY Salary ASC
    LIMIT 10
""").df()

,Salary,Quarter,Year,Age,Gender,Started,Profession,Grade,Localization,City,Skill,Industry,Created
0,58000,4,2024,24,Male,NaN,Дата инженер | Data Engineer,Middle,Local,Almaty,None,Банк,2024-10-11 13:05:57
1,75000,2,2025,20,Male,2025.0,Pentester,Junior,Local,Aktau,C/C++,HR системы,2025-05-31 14:12:27
2,75000,1,2025,27,Male,NaN,Android Developer,Trainee,Local,Almaty,None,E-Commerce,2025-02-05 09:20:51
3,75000,1,2025,20,Male,NaN,Copywriter,Trainee,Local,Aktobe,None,Продажи и маркетинг,2025-02-05 08:51:23
4,76000,3,2025,27,Female,2025.0,Talent Acquisition,Trainee,Local,Balqash,.NET,Нефть и газ,2025-08-16 09:20:26
5,85000,2,2025,21,PreferNotToSay,2021.0,Backend Developer,Middle,Local,Almaty,Python,Банк,2025-04-22 07:24:34
6,88888,2,2025,22,Male,2022.0,SecOps,Junior,Local,Aktau,Laravel,Телеком,2025-04-02 18:32:19
7,90000,4,2024,27,Male,NaN,DevOps,Trainee,Local,Almaty,None,Продажи и маркетинг,2024-10-28 16:45:32
8,100000,3,2025,20,Male,2024.0,Copywriter,Junior,Foreign,Aktau,.NET,Продажи и маркетинг,2025-08-05 19:08:04
9,100000,1,2025,24,Male,NaN,DevOps,Junior,Local,Almaty,.NET,Телеком,2025-02-07 10:59:59


In [154]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1366 entries, 0 to 1365
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Salary        1366 non-null   int64         
 1   Quarter       1366 non-null   int64         
 2   Year          1366 non-null   int64         
 3   Age           1366 non-null   int64         
 4   Gender        1366 non-null   object        
 5   Started       637 non-null    float64       
 6   Profession    1366 non-null   object        
 7   Grade         1366 non-null   object        
 8   Localization  1366 non-null   object        
 9   City          1366 non-null   object        
 10  Skill         879 non-null    object        
 11  Industry      1366 non-null   object        
 12  Created       1366 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(4), object(7)
memory usage: 138.9+ KB


In [155]:
# salary rank within each profession using DuckDB
conn.execute("""--sql
    SELECT 
        Profession,
        Grade,
        Salary,
        RANK() OVER (PARTITION BY Profession ORDER BY Salary DESC) AS rank_in_prof
    FROM salaries
    Limit 10
""").df()

,Profession,Grade,Salary,rank_in_prof
0,Android Developer,Senior,2300000,1
1,Android Developer,Lead,2100000,2
2,Android Developer,Senior,2100000,2
3,Android Developer,Middle,2000000,4
4,Android Developer,Senior,1800000,5
5,Android Developer,Middle,1677000,6
6,Android Developer,Senior,1500000,7
7,Android Developer,Senior,1500000,7
8,Android Developer,Senior,1450000,9
9,Android Developer,Junior,1430000,10


In [156]:
# salary dense rank within each profession using DuckDB
conn.execute("""--sql
    SELECT 
        Profession,
        Grade,
        Salary,
        DENSE_RANK() OVER (PARTITION BY Profession ORDER BY Salary DESC) AS rank_in_prof
    FROM salaries
    Limit 10
""").df()

,Profession,Grade,Salary,rank_in_prof
0,Chief Product Officer (CPO),Lead,2900000,1
1,Chief Product Officer (CPO),Lead,2500000,2
2,Chief Product Officer (CPO),Lead,2500000,2
3,Chief Product Officer (CPO),Lead,2000000,3
4,Chief Product Officer (CPO),Lead,1800000,4
5,Chief Product Officer (CPO),Lead,1500000,5
6,Chief Product Officer (CPO),Lead,1500000,5
7,Chief Product Officer (CPO),Middle,1400000,6
8,Chief Product Officer (CPO),Lead,1200000,7
9,Chief Product Officer (CPO),Senior,800000,8


In [157]:
# median salary per profession using window function in DuckDB
conn.execute("""--sql
    SELECT 
        Profession,
        cast(median(Salary) OVER (PARTITION BY Profession) as int) AS median_s_profession
    FROM salaries
    Order By median_s_profession DESC
    Limit 5;
""").df()

,Profession,median_s_profession
0,Security Engineer,3290000
1,Security Engineer,3290000
2,AI Engineer,2600000
3,Архитектор (Architect),2550000
4,Архитектор (Architect),2550000


In [158]:
# median salary per profession showing distinct rows
conn.execute("""--sql
    SELECT 
        Profession,
        CAST(median(Salary) AS INT) AS median_s_profession
    FROM salaries
    GROUP BY Profession
    ORDER BY median_s_profession DESC
    Limit 10;
""").df()

,Profession,median_s_profession
0,Security Engineer,3290000
1,AI Engineer,2600000
2,Архитектор (Architect),2550000
3,Computer vision engineer,2000000
4,CVM аналитик,2000000
5,Chief Technical Officer (CTO),2000000
6,Head Of Department,1900000
7,Delivery Manager,1900000
8,Скрам-мастер (Scrum Master),1900000
9,Техлид (Techleader),1850000


In [159]:
# median salary per profession showing distinct rows
conn.execute("""--sql
    SELECT 
        Profession,
        CAST(median(Salary) AS INT) AS median_s_profession
    FROM salaries
    GROUP BY Profession
    ORDER BY median_s_profession ASC
    Limit 10;
""").df()

,Profession,median_s_profession
0,SecOps,88888
1,Copywriter,100000
2,Talent Acquisition,180000
3,UI Designer,250000
4,IT Lector (Преподаватель),277500
5,Операционный аналитик,282000
6,Информационная безопасность,300000
7,DevRel (Developer Relations),300000
8,Data miner,300000
9,Customer Support,325000


In [160]:
# cumulative salary per profession using DuckDB (running total)
conn.execute("""--sql
    SELECT 
        Profession,
        Salary,
        cast(SUM(Salary) OVER (PARTITION BY Profession ORDER BY Salary DESC) as int) AS cumulative_salary
    FROM salaries
    Limit 10;
""").df()

,Profession,Salary,cumulative_salary
0,Chief Product Officer (CPO),2900000,2900000
1,Chief Product Officer (CPO),2500000,7900000
2,Chief Product Officer (CPO),2500000,7900000
3,Chief Product Officer (CPO),2000000,9900000
4,Chief Product Officer (CPO),1800000,11700000
5,Chief Product Officer (CPO),1500000,14700000
6,Chief Product Officer (CPO),1500000,14700000
7,Chief Product Officer (CPO),1400000,16100000
8,Chief Product Officer (CPO),1200000,17300000
9,Chief Product Officer (CPO),800000,18100000


In [161]:
df.head()

,Salary,Quarter,Year,Age,Gender,Started,Profession,Grade,Localization,City,Skill,Industry,Created
0,800000,3,2025,29,Male,2022.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,2025-09-01 17:08:55
1,500000,3,2025,19,Male,2024.0,Backend Developer,Junior,Foreign,Almaty,Java,IT продукт,2025-08-29 20:02:53
2,1160000,1,2025,23,Male,2022.0,Frontend Developer,Senior,Local,Almaty,Javascript,E-Commerce,2025-08-28 15:57:36
3,500000,3,2025,20,Male,2023.0,Backend Developer,Junior,Local,Astana,Java,Государственное учреждение (Government),2025-08-28 11:44:12
4,1200000,3,2025,33,Male,2017.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,2025-08-28 06:07:45


In [162]:
# Find the top 5 industries with the highest average salaries and list the top 3 professions within each of these industries based on average salary using DuckDB

conn.execute("""--sql

    WITH industry_avg AS (
        SELECT Industry, AVG(Salary) AS avg_salary
        FROM salaries
        GROUP BY Industry
        ORDER BY avg_salary DESC LIMIT 5),
    top_professions AS (
        SELECT s.Industry, s.Profession, AVG(s.Salary) AS avg_prof_salary,
        RANK() OVER (PARTITION BY s.Industry ORDER BY AVG(s.Salary) DESC) as prof_rank
        FROM salaries s
        JOIN industry_avg i ON s.Industry = i.Industry
        GROUP BY s.Industry, s.Profession)
    SELECT 
        Industry, Profession,
        CAST(avg_prof_salary AS INT) AS Avg_Profession_Salary,
        prof_rank
    FROM top_professions
    WHERE prof_rank <= 3
    ORDER BY Industry, prof_rank
    
""").df()

,Industry,Profession,Avg_Profession_Salary,prof_rank
0,E-Commerce,Архитектор (Architect),2600000,1
1,E-Commerce,Head Of Department,2250000,2
2,E-Commerce,Backend Developer,1640200,3
3,IT продукт,Security Engineer,3080000,1
4,IT продукт,Head Of Department,2406667,2
5,IT продукт,Chief Product Officer (CPO),2250000,3
6,Нефть и газ,Скрам-мастер (Scrum Master),2600000,1
7,Нефть и газ,1C Developer (1С разработчик),2300000,2
8,Нефть и газ,Data Scientist,1000000,3
9,Стартап,Data Scientist,3331000,1


In [163]:
# Top 5 industries by average salary, and top 3 professions within each industry
conn.execute("""--sql
    WITH industry_avg AS (
        SELECT Industry, AVG(Salary) AS avg_salary
        FROM salaries
        GROUP BY Industry
        ORDER BY avg_salary DESC LIMIT 5),
    top_professions AS (
        SELECT s.Industry, s.Profession, AVG(s.Salary) AS avg_prof_salary,
            DENSE_RANK() OVER (PARTITION BY s.Industry ORDER BY AVG(s.Salary) DESC) as prof_rank
        FROM salaries s
        JOIN industry_avg i ON s.Industry = i.Industry
        GROUP BY s.Industry, s.Profession)
    SELECT 
        Industry, Profession,
        CAST(avg_prof_salary AS INT) AS Avg_Profession_Salary, prof_rank
    FROM top_professions
    WHERE prof_rank <= 3
    ORDER BY Industry, prof_rank
""").df()

,Industry,Profession,Avg_Profession_Salary,prof_rank
0,E-Commerce,Архитектор (Architect),2600000,1
1,E-Commerce,Head Of Department,2250000,2
2,E-Commerce,Backend Developer,1640200,3
3,IT продукт,Security Engineer,3080000,1
4,IT продукт,Head Of Department,2406667,2
5,IT продукт,Chief Product Officer (CPO),2250000,3
6,Нефть и газ,Скрам-мастер (Scrum Master),2600000,1
7,Нефть и газ,1C Developer (1С разработчик),2300000,2
8,Нефть и газ,Data Scientist,1000000,3
9,Стартап,Data Scientist,3331000,1


In [164]:
# Calculate min, max, avg, median, and mode salary per Profession using DuckDB
conn.execute("""--sql
    SELECT 
        Profession,
        MIN(Salary) AS Min_Salary,
        MAX(Salary) AS Max_Salary,
        CAST(AVG(Salary) AS INT) AS Avg_Salary,
        CAST(median(Salary) AS INT) AS Median_Salary,
        mode() WITHIN GROUP (ORDER BY Salary) AS Mode_Salary
    FROM salaries
    GROUP BY Profession
    ORDER BY Avg_Salary DESC
    LIMIT 10
""").df()

,Profession,Min_Salary,Max_Salary,Avg_Salary,Median_Salary,Mode_Salary
0,Security Engineer,3080000,3500000,3290000,3290000,3080000
1,AI Engineer,2600000,2600000,2600000,2600000,2600000
2,Архитектор (Architect),1200000,2600000,2225000,2550000,2600000
3,CVM аналитик,2000000,2000000,2000000,2000000,2000000
4,Head Of Department,800000,3500000,1960159,1900000,1000000
5,Chief Technical Officer (CTO),445000,4000000,1904091,2000000,2500000
6,Скрам-мастер (Scrum Master),1200000,2600000,1900000,1900000,1200000
7,Delivery Manager,1200000,2500000,1866667,1900000,1200000
8,Техлид (Techleader),950000,2750000,1855667,1850000,950000
9,Chief Product Officer (CPO),800000,2900000,1810000,1650000,1500000


In [165]:
# Count, avg, min, max, median, mode, and frequency for "Data Analyst" profession using DuckDB
conn.execute("""--sql
    SELECT
        Profession,
        COUNT(*) AS Frequency,
        MIN(Salary) AS Min_Salary,
        MAX(Salary) AS Max_Salary,
        CAST(AVG(Salary) AS INTEGER) AS Avg_Salary,
        CAST(median(Salary) AS INTEGER) AS Median_Salary,
        mode() WITHIN GROUP (ORDER BY Salary) AS Mode_Salary
    FROM salaries
    WHERE Profession LIKE '%Data Analyst%'
    GROUP BY Profession
""").df()

,Profession,Frequency,Min_Salary,Max_Salary,Avg_Salary,Median_Salary,Mode_Salary
0,Дата аналитик | Data Analyst,103,120000,2280000,682788,560000,500000


In [166]:
# Average salary per Localization and profession where average salary > 2.000.000
conn.execute("""--sql
    SELECT 
        Localization, 
        Profession,
        cast(AVG(Salary) as int) AS avg_salary
    FROM salaries
    GROUP BY Localization, Profession
    HAVING AVG(Salary) > 2000000
    ORDER BY avg_salary DESC
    LIMIT 10
""").df()

,Localization,Profession,avg_salary
0,Local,Security Engineer,3290000
1,Foreign,Chief Technical Officer (CTO),3200000
2,Foreign,Техлид (Techleader),2734000
3,Foreign,AI Engineer,2600000
4,Foreign,Архитектор (Architect),2550000
5,Foreign,Product Manager,2294865
6,Foreign,Тимлид | Teamleader,2142857
7,Foreign,Pentester,2100000
8,Foreign,Data Warehouse Specialist | DWH,2058931
9,Local,Head Of Department,2008167


In [167]:
# Average age by gender where average age < 30
conn.execute("""--sql
    SELECT 
        Gender, 
        cast(AVG(Age) as int) AS avg_age
    FROM salaries
    GROUP BY Gender
    HAVING AVG(Age) < 30
""").df()

,Gender,avg_age
0,PreferNotToSay,26
1,Male,27
2,Female,28


In [168]:
# Get professions with their skills concatenated and count of skills per profession, ordered by count descending
conn.execute("""--sql--sql
    SELECT 
        Profession,
        string_agg(DISTINCT Skill, ', ') AS Skills,
        COUNT(DISTINCT Skill) AS Skill_Count
    FROM salaries
    WHERE Skill IS NOT NULL
    GROUP BY Profession
    ORDER BY Skill_Count DESC
    LIMIT 10
""").df()

,Profession,Skills,Skill_Count
0,Backend Developer,"Kotlin, Golang, Laravel, Javascript, Другое, J...",19
1,Fullstack Developer,".NET, Node.js, Python, Vue, Ruby, FastAPI, PHP...",15
2,Quality Assurance (QA),"Python, C/C++, .NET, Swift, Kotlin, Java, Pyte...",13
3,Тимлид | Teamleader,"Java, Javascript, Laravel, Golang, Другое, Mac...",12
4,Frontend Developer,"React, Typescript, Java, Javascript, Angular, ...",10
5,Системный аналитик | System Analyst,"Дата аналитика | Data Analysis, Kotlin, BI, Др...",9
6,Chief Technical Officer (CTO),"Laravel, Golang, Java, Python, .NET, Typescrip...",7
7,Business Analyst (BA),"ERP Systems, 1C, Другое, Python, BI, Дата анал...",6
8,DevOps,"Другое, Golang, .NET, C/C++, Python, Data Ware...",6
9,Техлид (Techleader),"Node.js, Другое, Laravel, Golang, Java",5


In [169]:
# cumulative salary per profession using DuckDB (running total)
conn.execute("""--sql--sql
    SELECT 
        Profession,
        Salary,
        SUM(Salary) OVER (PARTITION BY Profession ORDER BY Salary) AS cumulative_salary,
        SUM(Salary) OVER (PARTITION BY Profession) AS total_salary
    FROM salaries
    LIMIT 10
""").df()

,Profession,Salary,cumulative_salary,total_salary
0,Chief Product Officer (CPO),800000,800000.0,18100000.0
1,Chief Product Officer (CPO),1200000,2000000.0,18100000.0
2,Chief Product Officer (CPO),1400000,3400000.0,18100000.0
3,Chief Product Officer (CPO),1500000,6400000.0,18100000.0
4,Chief Product Officer (CPO),1500000,6400000.0,18100000.0
5,Chief Product Officer (CPO),1800000,8200000.0,18100000.0
6,Chief Product Officer (CPO),2000000,10200000.0,18100000.0
7,Chief Product Officer (CPO),2500000,15200000.0,18100000.0
8,Chief Product Officer (CPO),2500000,15200000.0,18100000.0
9,Chief Product Officer (CPO),2900000,18100000.0,18100000.0


In [170]:
# Calculate min, max, avg, mode, median, and range salary per Grade using DuckDB
conn.execute("""--sql
    SELECT 
        Grade,
        MIN(Salary) AS Min_Salary,
        MAX(Salary) AS Max_Salary,
        CAST(AVG(Salary) AS INTEGER) AS Avg_Salary,
        CAST(median(Salary) AS INTEGER) AS Median_Salary,
        mode(Salary) AS Mode_Salary,
        (MAX(Salary) - MIN(Salary)) AS Salary_Range
    FROM salaries
    GROUP BY Grade
    ORDER BY Avg_Salary DESC
""").df()

,Grade,Min_Salary,Max_Salary,Avg_Salary,Median_Salary,Mode_Salary,Salary_Range
0,Lead,100000,4000000,1648205,1500000,1500000,3900000
1,Senior,150000,5400000,1452603,1292500,1000000,5250000
2,Middle,58000,4000000,884842,800000,800000,3942000
3,Junior,75000,1500000,395395,350000,300000,1425000
4,Trainee,75000,500000,199417,175000,200000,425000


In [171]:
# cumulative distribution of salaries within each profession using DuckDB
conn.execute("""--sql--sql
    SELECT
        Profession,
        Salary,
        CUME_DIST() OVER (PARTITION BY Profession ORDER BY Salary) AS cume_dist
    FROM salaries
    LIMIT 10
""").df()

,Profession,Salary,cume_dist
0,Chief Product Officer (CPO),800000,0.1
1,Chief Product Officer (CPO),1200000,0.2
2,Chief Product Officer (CPO),1400000,0.3
3,Chief Product Officer (CPO),1500000,0.5
4,Chief Product Officer (CPO),1500000,0.5
5,Chief Product Officer (CPO),1800000,0.6
6,Chief Product Officer (CPO),2000000,0.7
7,Chief Product Officer (CPO),2500000,0.9
8,Chief Product Officer (CPO),2500000,0.9
9,Chief Product Officer (CPO),2900000,1.0


In [172]:
# Lead function to get next salary per profession using DuckDB
conn.execute("""--sql--sql
    SELECT 
        Profession,
        Salary,
        LEAD(Salary) OVER (PARTITION BY Profession ORDER BY Salary DESC) AS next_salary
    FROM salaries
    LIMIT 10
""").df()

,Profession,Salary,next_salary
0,Android Developer,2300000,2100000
1,Android Developer,2100000,2100000
2,Android Developer,2100000,2000000
3,Android Developer,2000000,1800000
4,Android Developer,1800000,1677000
5,Android Developer,1677000,1500000
6,Android Developer,1500000,1500000
7,Android Developer,1500000,1450000
8,Android Developer,1450000,1430000
9,Android Developer,1430000,1400000


In [173]:
# Lag function to get previous salary per grade using DuckDB
conn.execute("""--sql
    SELECT 
        Grade,
        Salary,
        LAG(Salary) OVER (PARTITION BY Grade ORDER BY Salary DESC) AS prev_salary
    FROM salaries
    LIMIT 10
""").df()

,Grade,Salary,prev_salary
0,Middle,4000000,<NA>
1,Middle,3500000,4000000
2,Middle,3000000,3500000
3,Middle,2600000,3000000
4,Middle,2400000,2600000
5,Middle,2400000,2400000
6,Middle,2334000,2400000
7,Middle,2200000,2334000
8,Middle,2100000,2200000
9,Middle,2000000,2100000


In [174]:
# FIRST_Value function to get minimum salary per profession (if ordered descending, it gives the maximum, if ordered ascending, it gives the minimum)
conn.execute("""--sql--sql
    SELECT 
        Profession,
        Salary,
        FIRST_Value(Salary) OVER (PARTITION BY Profession ORDER BY Salary ASC) AS min_salary
    FROM salaries
    LIMIT 10
""").df()

,Profession,Salary,min_salary
0,Product Analyst,150000,150000
1,Product Analyst,480000,150000
2,Product Analyst,700000,150000
3,Product Analyst,800000,150000
4,Product Analyst,1000000,150000
5,Product Analyst,1000000,150000
6,Product Analyst,1400000,150000
7,Product Analyst,1400000,150000
8,Product Analyst,1500000,150000
9,UX Designer,500000,500000


In [175]:
# Last_Value function to get maximum salary per profession using DuckDB
conn.execute("""--sql--sql
    SELECT 
        Profession,
        Salary,
        LAST_Value(Salary) OVER (PARTITION BY Profession ORDER BY Salary ASC 
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS max_salary
    FROM salaries
    LIMIT 10
""").df()

,Profession,Salary,max_salary
0,1C Developer (1С разработчик),600000,2300000
1,1C Developer (1С разработчик),650000,2300000
2,1C Developer (1С разработчик),1200000,2300000
3,1C Developer (1С разработчик),1320000,2300000
4,1C Developer (1С разработчик),2000000,2300000
5,1C Developer (1С разработчик),2300000,2300000
6,Data Scientist,150000,3331000
7,Data Scientist,280000,3331000
8,Data Scientist,450000,3331000
9,Data Scientist,500000,3331000


In [176]:
# NTH_Value function to get the 3rd highest salary per Profession
conn.execute("""--sql
    SELECT 
        Profession,
        Salary,
        NTH_Value(Salary, 3) OVER (PARTITION BY Profession ORDER BY Salary DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS third_salary
    FROM salaries
    LIMIT 10
""").df()

,Profession,Salary,third_salary
0,Product Analyst,1500000,1400000
1,Product Analyst,1400000,1400000
2,Product Analyst,1400000,1400000
3,Product Analyst,1000000,1400000
4,Product Analyst,1000000,1400000
5,Product Analyst,800000,1400000
6,Product Analyst,700000,1400000
7,Product Analyst,480000,1400000
8,Product Analyst,150000,1400000
9,UX Designer,600000,<NA>


In [177]:
df.head()

,Salary,Quarter,Year,Age,Gender,Started,Profession,Grade,Localization,City,Skill,Industry,Created
0,800000,3,2025,29,Male,2022.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,2025-09-01 17:08:55
1,500000,3,2025,19,Male,2024.0,Backend Developer,Junior,Foreign,Almaty,Java,IT продукт,2025-08-29 20:02:53
2,1160000,1,2025,23,Male,2022.0,Frontend Developer,Senior,Local,Almaty,Javascript,E-Commerce,2025-08-28 15:57:36
3,500000,3,2025,20,Male,2023.0,Backend Developer,Junior,Local,Astana,Java,Государственное учреждение (Government),2025-08-28 11:44:12
4,1200000,3,2025,33,Male,2017.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,2025-08-28 06:07:45


In [178]:
# Count Grades within every Gender and show average salary
conn.execute("""--sql
    SELECT 
        Gender,
        Grade,
        COUNT(*) AS Grade_Count,
        CAST(AVG(Salary) AS INT) AS Avg_Salary,
        SUM(COUNT(*)) OVER (PARTITION BY Gender) AS Total_Grade_Count
    FROM salaries
    GROUP BY Gender, Grade
    ORDER BY 
        Total_Grade_Count DESC,  -- gender with highest total first
        Avg_Salary DESC;         -- then by average salary
   
""").df()

,Gender,Grade,Grade_Count,Avg_Salary,Total_Grade_Count
0,Male,Lead,125,1710467,1078.0
1,Male,Senior,238,1557741,1078.0
2,Male,Middle,498,897669,1078.0
3,Male,Junior,198,387843,1078.0
4,Male,Trainee,19,196316,1078.0
5,Female,Lead,25,1308440,259.0
6,Female,Senior,60,1046400,259.0
7,Female,Middle,122,823451,259.0
8,Female,Junior,47,428489,259.0
9,Female,Trainee,5,211200,259.0


In [179]:
# Count number of entries per Profession and Grade using DuckDB
conn.execute("""--sql
    SELECT DISTINCT --if you want unique combinations
        Profession,
        Grade,
        COUNT(*) OVER (PARTITION BY Profession, Grade) AS count_per_group
    FROM salaries
    OrDER BY count_per_group DESC
    LIMIT 10
""").df()

,Profession,Grade,count_per_group
0,Backend Developer,Middle,151
1,Frontend Developer,Middle,78
2,Backend Developer,Junior,64
3,Backend Developer,Senior,55
4,Дата аналитик | Data Analyst,Middle,41
5,Frontend Developer,Senior,41
6,Тимлид | Teamleader,Lead,40
7,Quality Assurance (QA),Middle,33
8,Дата инженер | Data Engineer,Middle,32
9,Frontend Developer,Junior,32


In [180]:
# Count number of entries per Profession and Grade using DuckDB
conn.execute("""--sql
    SELECT 
        Profession,
        Grade,
        COUNT(*) AS count_per_group
    FROM salaries
    GROUP BY Profession, Grade
    ORDER BY count_per_group DESC
    LIMIT 10
""").df()

,Profession,Grade,count_per_group
0,Backend Developer,Middle,151
1,Frontend Developer,Middle,78
2,Backend Developer,Junior,64
3,Backend Developer,Senior,55
4,Дата аналитик | Data Analyst,Middle,41
5,Frontend Developer,Senior,41
6,Тимлид | Teamleader,Lead,40
7,Quality Assurance (QA),Middle,33
8,Frontend Developer,Junior,32
9,Дата инженер | Data Engineer,Middle,32


In [181]:
# Count number of entries per Profession and Grade using DuckDB, ordered by total per Profession and then within Profession
conn.execute("""--sql
    SELECT 
        Profession,
        Grade,
        COUNT(*) AS count_per_group
    FROM salaries
    GROUP BY Profession, Grade
    ORDER BY 
        SUM(count_per_group) OVER (PARTITION BY Profession) DESC,  -- total per Profession
        count_per_group DESC                                -- then within Profession
    LIMIT 20
""").df()

,Profession,Grade,count_per_group
0,Backend Developer,Middle,151
1,Backend Developer,Junior,64
2,Backend Developer,Senior,55
3,Backend Developer,Lead,9
4,Backend Developer,Trainee,1
5,Frontend Developer,Middle,78
6,Frontend Developer,Senior,41
7,Frontend Developer,Junior,32
8,Frontend Developer,Lead,7
9,Frontend Developer,Trainee,6


In [182]:
# Count number of entries per Profession and Grade using DuckDB
conn.execute("""--sql
    SELECT DISTINCT
        Profession,
        Grade,
        count_per_group
    FROM (
        SELECT 
            Profession,
            Grade,
            COUNT(*) OVER (PARTITION BY Profession, Grade) AS count_per_group
        FROM salaries)
    ORDER BY
        SUM(count_per_group) OVER (PARTITION BY Profession) DESC,  -- total per Profession 
        count_per_group DESC
    LIMIT 10
""").df()

,Profession,Grade,count_per_group
0,Backend Developer,Middle,151
1,Backend Developer,Junior,64
2,Backend Developer,Senior,55
3,Backend Developer,Lead,9
4,Backend Developer,Trainee,1
5,Frontend Developer,Middle,78
6,Frontend Developer,Senior,41
7,Frontend Developer,Junior,32
8,Frontend Developer,Lead,7
9,Frontend Developer,Trainee,6


In [183]:
numbers = [1, 2, 5, 6, 10]

# Average (mean)
average = sum(numbers) / len(numbers)

# Median
sorted_numbers = sorted(numbers)
n = len(sorted_numbers)
if n % 2 == 1:
    median = sorted_numbers[n // 2]
else:
    median = (sorted_numbers[n // 2 - 1] + sorted_numbers[n // 2]) / 2

# Mode (moda)
counts = {}
for num in numbers:
    counts[num] = counts.get(num, 0) + 1
max_count = max(counts.values())
mode = [k for k, v in counts.items() if v == max_count]

print("Average:", average)
print("Median:", median)
print("Mode:", mode)

Average: 4.8
Median: 5
Mode: [1, 2, 5, 6, 10]
